# Fit PCA

> Fit PCA on pre-encoded embeddings and save reduced representations for generative model training.

In [ ]:
#| default_exp fit_pca

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import torch
import pickle
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
from sklearn.decomposition import PCA
from omegaconf import DictConfig
import hydra

In [ ]:
#| export
def load_level_embeddings(encoded_dir: Path, split: str, level: int = 0, key: str = 'emb2',
                          max_rows: int = None):
    """Load patch embeddings at a given hierarchy level from pre-encoded chunk files.
    Returns (N_total * N_patches, D) as float16 numpy array — caller converts to float32 before fitting.
    max_rows: if set, subsample proportionally per chunk during loading to cap peak RAM usage.
    key: 'emb1', 'emb2', or 'emb3' (which image from the triplet)."""
    import numpy as np
    chunks = sorted(encoded_dir.glob(f"{split}_chunk*.pt"))
    assert chunks, f"No chunks found for split={split} in {encoded_dir}"
    all_embs = []
    for chunk_path in tqdm(chunks, desc=f"Loading {split} L{level}"):
        chunk = torch.load(chunk_path, weights_only=False)
        parts = []
        for batch_rec in chunk:
            emb = batch_rec[key][level]          # (B, N_patches, D) float16
            B, P, D = emb.shape
            parts.append(emb.reshape(B * P, D).numpy())   # stay float16
        chunk_emb = np.concatenate(parts, axis=0)          # float16
        if max_rows is not None:
            frac = max_rows / (len(chunks) * chunk_emb.shape[0] + 1)
            if frac < 1.0:
                idx = np.random.choice(chunk_emb.shape[0], max(1, int(chunk_emb.shape[0] * frac)), replace=False)
                chunk_emb = chunk_emb[idx]
        all_embs.append(chunk_emb)
    result = np.concatenate(all_embs, axis=0)              # float16
    if max_rows is not None and result.shape[0] > max_rows:
        idx = np.random.choice(result.shape[0], max_rows, replace=False)
        result = result[idx]
    return result  # float16; caller does .astype(np.float32) before sklearn PCA


In [ ]:
#| export
def _wipe_pca_dir(output_dir):
    "Remove stale PKL and projected chunk files before a fresh fit."
    stale = list(output_dir.glob("pca_L*.pkl")) + list(output_dir.glob("*_pca.pt"))
    if stale:
        print(f"Removing {len(stale)} stale PCA files from {output_dir}...")
        for f in stale: f.unlink()

def _fit_one_level(encoded_dir, output_dir, level, n_comp, key, max_fit_rows):
    """Fit PCA for one level. n_comp: int (fixed count) or float in (0,1) (variance target).
    Saves pca_L{level}_n{actual_n}.pkl. Returns (pca_object, actual_n)."""
    import numpy as np
    sample = torch.load(sorted(encoded_dir.glob("train_chunk*.pt"))[0], weights_only=False)
    D = sample[0][key][level].shape[-1]
    is_var = isinstance(n_comp, float) and 0.0 < n_comp < 1.0
    if not is_var and int(n_comp) >= D:
        print(f"L{level}: D={D} ≤ n_comp={n_comp}, saving raw")
        return None, D
    n = n_comp if is_var else min(int(n_comp), D)
    print(f"Fitting PCA(n={'%.0f%%'%(n*100) if is_var else n}) on L{level} (D={D})...")
    emb = load_level_embeddings(encoded_dir, "train", level=level, key=key, max_rows=max_fit_rows)
    pca = PCA(n_components=n, whiten=False)
    pca.fit(emb.astype(np.float32))
    actual_n, var = pca.n_components_, pca.explained_variance_ratio_.cumsum()[-1]
    print(f"  L{level}: {actual_n} components → {var:.1%} variance")
    with open(output_dir / f"pca_L{level}_n{actual_n}.pkl", "wb") as f: pickle.dump(pca, f)
    return pca, actual_n

def _project_chunks(encoded_dir, output_dir, levels, pcas, key, split):
    "Project all chunks for one split through fitted PCA models; save *_pca.pt files."
    chunks = sorted(encoded_dir.glob(f"{split}_chunk*.pt"))
    print(f"Projecting {len(chunks)} {split} chunks...")
    for chunk_path in tqdm(chunks, desc=split):
        data = torch.load(chunk_path, weights_only=False)
        out = {}
        for level in levels:
            embs = torch.cat([rec[key][level].float() for rec in data], dim=0)
            N, P, D = embs.shape
            if pcas[level] is None:
                out[f"L{level}"] = embs
            else:
                flat = embs.reshape(N*P, D).numpy()
                proj = torch.tensor(pcas[level].transform(flat), dtype=torch.float32).reshape(N, P, -1)
                out[f"L{level}"] = proj
        torch.save(out, output_dir / f"{chunk_path.stem}_pca.pt")
    print(f"  done → {output_dir}")

In [ ]:
#| export
def fit_and_save_pca(cfg):
    "Wipe stale files, fit PCA per level, project all chunks. n_per_lvl entries: int=fixed, float<1=variance target."
    fc = cfg.fitpca
    encoded_dir = Path(os.path.expandvars(cfg.preencode.output_dir))
    output_dir  = Path(os.path.expandvars(fc.output_dir))
    output_dir.mkdir(parents=True, exist_ok=True)
    key = "embeddings"

    _wipe_pca_dir(output_dir)

    levels  = list(fc.levels)
    n_comps = list(fc.n_per_lvl)
    pcas = {}
    for level, n_comp in zip(levels, n_comps):
        pca, _ = _fit_one_level(encoded_dir, output_dir, level, n_comp, key, max_fit_rows=None)
        pcas[level] = pca

    for split in ("train", "val"):
        _project_chunks(encoded_dir, output_dir, levels, pcas, key, split)

## Interactive usage

In [ ]:
#| eval: false
# Example: fit PCA(32) on L0 embeddings
pca = fit_and_save_pca(
    encoded_dir='~/datasets/POP909_encoded',
    output_dir='~/datasets/POP909_pca',
    level=0,
    n_components=32,
)

In [ ]:
#| export
#| eval: false
@hydra.main(version_base=None, config_path="../configs", config_name="config_swin")
def fit_pca_main(cfg: DictConfig):
    fit_and_save_pca(cfg)
    print("FINISHED")

if __name__ == '__main__' and 'ipykernel' not in __import__('sys').modules:
    fit_pca_main()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()